In [1]:
import numpy as np
import pandas as pd
import os
import kagglehub

# Only print folder structure, not all files
for dirname, dirs, filenames in os.walk('/kaggle/input'):
    print(dirname, "-> files:", len(filenames))

/kaggle/input -> files: 0
/kaggle/input/datasets -> files: 0
/kaggle/input/datasets/shreyaty08 -> files: 0
/kaggle/input/datasets/shreyaty08/fakeavceleb -> files: 0
/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2 -> files: 1
/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/moved -> files: 1
/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2 -> files: 3
/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/FakeVideo-FakeAudio -> files: 1
/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/African -> files: 1
/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/African/men -> files: 1
/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/FakeVideo-FakeAudio/African/men/id00391 -> files: 24
/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/FakeVideo-FakeAudi

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
import os
import pandas as pd

base = '/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2'
categories = ['RealVideo-RealAudio', 'FakeVideo-FakeAudio', 'FakeVideo-RealAudio', 'RealVideo-FakeAudio']

data = []
for cat in categories:
    cat_path = os.path.join(base, cat)
    for ethnicity in os.listdir(cat_path):
        eth_path = os.path.join(cat_path, ethnicity)
        if not os.path.isdir(eth_path):
            continue
        for gender in os.listdir(eth_path):
            gen_path = os.path.join(eth_path, gender)
            if not os.path.isdir(gen_path):
                continue
            for identity in os.listdir(gen_path):
                id_path = os.path.join(gen_path, identity)
                if not os.path.isdir(id_path):
                    continue
                for f in os.listdir(id_path):
                    if f.endswith('.mp4'):
                        label = 'real' if cat == 'RealVideo-RealAudio' else 'fake'
                        data.append({
                            'path': os.path.join(id_path, f),
                            'category': cat,
                            'ethnicity': ethnicity,
                            'gender': gender,
                            'identity': identity,
                            'label': label
                        })

df = pd.DataFrame(data)
print(df.shape)
print(df['label'].value_counts())
print(df['category'].value_counts())
df.to_csv('/kaggle/working/fakeavceleb_index.csv', index=False)

(21560, 6)
label
fake    21060
real      500
Name: count, dtype: int64
category
FakeVideo-FakeAudio    10851
FakeVideo-RealAudio     9709
RealVideo-RealAudio      500
RealVideo-FakeAudio      500
Name: count, dtype: int64


In [4]:
import os
import cv2
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [5]:
df = pd.read_csv('/kaggle/working/fakeavceleb_index.csv')

# Reduce to 300 real + 600 fake only (smaller, fits in memory)
real_df = df[df['label'] == 'real'].sample(n=300, random_state=42)
fake_df = df[df['label'] == 'fake'].sample(n=600, random_state=42)
df_balanced = pd.concat([real_df, fake_df]).sample(frac=1, random_state=42).reset_index(drop=True)

df_balanced['label_int'] = (df_balanced['label'] == 'fake').astype(int)
print(df_balanced['label'].value_counts())

train_df, temp_df = train_test_split(df_balanced, test_size=0.3, stratify=df_balanced['label_int'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label_int'], random_state=42)
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

label
fake    600
real    300
Name: count, dtype: int64
Train: 630, Val: 135, Test: 135


In [6]:
class FakeAVCelebDataset(Dataset):
    def __init__(self, df, frame_transform=None, max_frames=5):
        self.df = df.reset_index(drop=True)
        self.frame_transform = frame_transform
        self.max_frames = max_frames
        self.frames_base = '/kaggle/input/datasets/shreyaty08/fakeavceleb/FakeAVCeleb_v1.2/frames'

    def __len__(self):
        return len(self.df)

    def get_frames(self, video_path):
        # video_path like: .../FakeAVCeleb_v1.2/FakeVideo-FakeAudio/African/men/id00225/xxxxx.mp4
        # frames folder:   .../frames/FakeVideo-FakeAudio/African/men/id00225/
        after = video_path.split('/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2/')[-1]  # cat/eth/gender/id/file.mp4
        folder_rel = '/'.join(after.split('/')[:-1])                           # cat/eth/gender/id
        frame_dir = os.path.join(self.frames_base, folder_rel)

        frames = []
        if os.path.isdir(frame_dir):
            files = sorted([f for f in os.listdir(frame_dir)
                           if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
            if files:
                indices = np.linspace(0, len(files)-1, self.max_frames, dtype=int)
                for i in indices:
                    try:
                        from PIL import Image
                        img = Image.open(os.path.join(frame_dir, files[i])).convert('RGB')
                        if self.frame_transform:
                            frames.append(self.frame_transform(img))
                        else:
                            import torchvision.transforms.functional as TF
                            frames.append(TF.to_tensor(img))
                    except:
                        frames.append(torch.zeros(3, 112, 112))

        while len(frames) < self.max_frames:
            frames.append(torch.zeros(3, 112, 112))
        return torch.stack(frames[:self.max_frames])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        frames = self.get_frames(row['path'])
        label  = torch.tensor(row['label_int'], dtype=torch.long)
        return frames, label

frame_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = FakeAVCelebDataset(train_df, frame_transform=frame_transform)
val_ds   = FakeAVCelebDataset(val_df,   frame_transform=frame_transform)
test_ds  = FakeAVCelebDataset(test_df,  frame_transform=frame_transform)

# Quick sanity check — print one sample's frame shape
frames, label = train_ds[0]
print(f"Frame shape: {frames.shape}, Label: {label}")  # should be (5, 3, 112, 112)

train_labels = train_df['label_int'].values
class_counts = np.bincount(train_labels)
weights = 1.0 / class_counts[train_labels]
sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False,  num_workers=2, pin_memory=True)
print("Dataloaders ready.")

Frame shape: torch.Size([5, 3, 112, 112]), Label: 1
Dataloaders ready.


In [7]:
class VideoOnlyDetector(nn.Module):
    """ResNet18 backbone — video only baseline"""
    def __init__(self, embed_dim=256, num_classes=2):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, frames):
        B, T, C, H, W = frames.shape
        x = frames.view(B * T, C, H, W)
        feat = self.backbone(x).squeeze(-1).squeeze(-1)  # (B*T, 512)
        feat = feat.view(B, T, 512).mean(dim=1)          # (B, 512) temporal mean
        return self.classifier(feat)

model = VideoOnlyDetector().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 148MB/s]


Trainable parameters: 11,308,354


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for frames, labels in loader:
        frames, labels = frames.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(frames)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs, all_labels = [], []
    with torch.no_grad():
        for frames, labels in loader:
            frames, labels = frames.to(device), labels.to(device)
            out = model(frames)
            loss = criterion(out, labels)
            total_loss += loss.item()
            probs = F.softmax(out, dim=1)[:, 1]
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader), correct / total, auc

EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}
best_auc = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc, vl_auc = eval_epoch(model, val_loader, criterion)
    scheduler.step(vl_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['val_auc'].append(vl_auc)

    print(f"Epoch {epoch:02d} | Train Loss: {tr_loss:.4f}, Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}, Acc: {vl_acc:.4f}, AUC: {vl_auc:.4f}")

    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model.state_dict(), '/kaggle/working/video_only_resnet18_best.pth')
        print(f"  ✓ Best model saved (AUC: {best_auc:.4f})")

Epoch 01 | Train Loss: 0.6219, Acc: 0.6524 | Val Loss: 0.7595, Acc: 0.6148, AUC: 0.5899
  ✓ Best model saved (AUC: 0.5899)
Epoch 02 | Train Loss: 0.5188, Acc: 0.7968 | Val Loss: 0.6742, Acc: 0.6519, AUC: 0.7630
  ✓ Best model saved (AUC: 0.7630)
Epoch 03 | Train Loss: 0.3245, Acc: 0.8683 | Val Loss: 1.2364, Acc: 0.7259, AUC: 0.7440
Epoch 04 | Train Loss: 0.2548, Acc: 0.9159 | Val Loss: 0.8509, Acc: 0.7407, AUC: 0.8072
  ✓ Best model saved (AUC: 0.8072)
Epoch 05 | Train Loss: 0.2327, Acc: 0.9270 | Val Loss: 1.0434, Acc: 0.8074, AUC: 0.8493
  ✓ Best model saved (AUC: 0.8493)
Epoch 06 | Train Loss: 0.1831, Acc: 0.9571 | Val Loss: 0.8736, Acc: 0.8519, AUC: 0.8447
Epoch 07 | Train Loss: 0.1565, Acc: 0.9635 | Val Loss: 1.3655, Acc: 0.8000, AUC: 0.8443
Epoch 08 | Train Loss: 0.1353, Acc: 0.9587 | Val Loss: 1.0417, Acc: 0.8148, AUC: 0.8627
  ✓ Best model saved (AUC: 0.8627)
Epoch 09 | Train Loss: 0.0860, Acc: 0.9794 | Val Loss: 0.9796, Acc: 0.8444, AUC: 0.8715
  ✓ Best model saved (AUC: 0.8715

# Xception

In [9]:
class XceptionStream(nn.Module):
    """Xception-style video stream — widely used for deepfake/face forgery detection"""
    def __init__(self, embed_dim=256, num_classes=2):
        super().__init__()
        import timm
        self.backbone = timm.create_model('xception', pretrained=True, num_classes=0)  # feature extractor
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, frames):
        B, T, C, H, W = frames.shape
        x = frames.view(B * T, C, H, W)
        feat = self.backbone(x)               # (B*T, feat_dim)
        feat = feat.view(B, T, -1).mean(dim=1) # (B, feat_dim) temporal mean
        return self.classifier(feat)

model_xception = XceptionStream().to(device)
total_params = sum(p.numel() for p in model_xception.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
Trainable parameters: 21,332,010


In [10]:
!pip install timm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_xception.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

EPOCHS = 10
history_xception = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}
best_auc = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model_xception, train_loader, optimizer, criterion)
    vl_loss, vl_acc, vl_auc = eval_epoch(model_xception, val_loader, criterion)
    scheduler.step(vl_loss)

    history_xception['train_loss'].append(tr_loss)
    history_xception['val_loss'].append(vl_loss)
    history_xception['train_acc'].append(tr_acc)
    history_xception['val_acc'].append(vl_acc)
    history_xception['val_auc'].append(vl_auc)

    print(f"Epoch {epoch:02d} | Train Loss: {tr_loss:.4f}, Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}, Acc: {vl_acc:.4f}, AUC: {vl_auc:.4f}")

    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model_xception.state_dict(), '/kaggle/working/xception_best.pth')
        print(f"  ✓ Best model saved (AUC: {best_auc:.4f})")

Epoch 01 | Train Loss: 0.6795, Acc: 0.5254 | Val Loss: 0.7596, Acc: 0.5481, AUC: 0.4933
  ✓ Best model saved (AUC: 0.4933)
Epoch 02 | Train Loss: 0.6405, Acc: 0.6476 | Val Loss: 0.7790, Acc: 0.6444, AUC: 0.5499
  ✓ Best model saved (AUC: 0.5499)
Epoch 03 | Train Loss: 0.5276, Acc: 0.7508 | Val Loss: 1.2112, Acc: 0.7037, AUC: 0.6590
  ✓ Best model saved (AUC: 0.6590)
Epoch 04 | Train Loss: 0.3833, Acc: 0.8619 | Val Loss: 3.7711, Acc: 0.7185, AUC: 0.6969
  ✓ Best model saved (AUC: 0.6969)
Epoch 05 | Train Loss: 0.2603, Acc: 0.9159 | Val Loss: 3.2047, Acc: 0.7481, AUC: 0.7143
  ✓ Best model saved (AUC: 0.7143)
Epoch 06 | Train Loss: 0.2213, Acc: 0.9286 | Val Loss: 3.7855, Acc: 0.7704, AUC: 0.7251
  ✓ Best model saved (AUC: 0.7251)
Epoch 07 | Train Loss: 0.1885, Acc: 0.9540 | Val Loss: 3.1805, Acc: 0.8148, AUC: 0.7714
  ✓ Best model saved (AUC: 0.7714)
Epoch 08 | Train Loss: 0.1669, Acc: 0.9460 | Val Loss: 1.6545, Acc: 0.8296, AUC: 0.7932
  ✓ Best model saved (AUC: 0.7932)
Epoch 09 | Train

# EfficientNet-B0

In [12]:
import torch.nn as nn

class EfficientNetStream(nn.Module):
    """EfficientNet-B0 — lightweight, strong baseline for deepfake detection"""
    def __init__(self, num_classes=2):
        super().__init__()
        import timm
        self.backbone = timm.create_model('efficientnet_b0', pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, frames):
        B, T, C, H, W = frames.shape
        x = frames.view(B * T, C, H, W)
        feat = self.backbone(x)
        feat = feat.view(B, T, -1).mean(dim=1)
        return self.classifier(feat)

model_effnet = EfficientNetStream().to(device)
total_params = sum(p.numel() for p in model_effnet.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Trainable parameters: 4,335,998


In [13]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_effnet.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

EPOCHS = 10
history_effnet = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}
best_auc = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model_effnet, train_loader, optimizer, criterion)
    vl_loss, vl_acc, vl_auc = eval_epoch(model_effnet, val_loader, criterion)
    scheduler.step(vl_loss)

    history_effnet['train_loss'].append(tr_loss)
    history_effnet['val_loss'].append(vl_loss)
    history_effnet['train_acc'].append(tr_acc)
    history_effnet['val_acc'].append(vl_acc)
    history_effnet['val_auc'].append(vl_auc)

    print(f"Epoch {epoch:02d} | Train Loss: {tr_loss:.4f}, Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}, Acc: {vl_acc:.4f}, AUC: {vl_auc:.4f}")

    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model_effnet.state_dict(), '/kaggle/working/efficientnet_best.pth')
        print(f"  ✓ Best model saved (AUC: {best_auc:.4f})")

Epoch 01 | Train Loss: 0.6217, Acc: 0.6968 | Val Loss: 0.5084, Acc: 0.8000, AUC: 0.8825
  ✓ Best model saved (AUC: 0.8825)
Epoch 02 | Train Loss: 0.3122, Acc: 0.9048 | Val Loss: 0.2244, Acc: 0.8963, AUC: 0.9686
  ✓ Best model saved (AUC: 0.9686)
Epoch 03 | Train Loss: 0.2125, Acc: 0.9206 | Val Loss: 0.1683, Acc: 0.9407, AUC: 0.9798
  ✓ Best model saved (AUC: 0.9798)
Epoch 04 | Train Loss: 0.1384, Acc: 0.9587 | Val Loss: 0.1257, Acc: 0.9704, AUC: 0.9842
  ✓ Best model saved (AUC: 0.9842)
Epoch 05 | Train Loss: 0.2304, Acc: 0.9460 | Val Loss: 0.1472, Acc: 0.9630, AUC: 0.9830
Epoch 06 | Train Loss: 0.1559, Acc: 0.9444 | Val Loss: 0.1807, Acc: 0.9630, AUC: 0.9812
Epoch 07 | Train Loss: 0.0859, Acc: 0.9762 | Val Loss: 0.1723, Acc: 0.9704, AUC: 0.9822
Epoch 08 | Train Loss: 0.1022, Acc: 0.9730 | Val Loss: 0.1159, Acc: 0.9778, AUC: 0.9886
  ✓ Best model saved (AUC: 0.9886)
Epoch 09 | Train Loss: 0.0752, Acc: 0.9714 | Val Loss: 0.1634, Acc: 0.9556, AUC: 0.9864
Epoch 10 | Train Loss: 0.1436, Ac

# ViT model

In [14]:
import torch.nn as nn

class ViTStream(nn.Module):
    """Vision Transformer — captures global patch relationships, no convolutions"""
    def __init__(self, num_classes=2):
        super().__init__()
        import timm
        self.backbone = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=0)
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )

    def forward(self, frames):
        B, T, C, H, W = frames.shape
        x = frames.view(B * T, C, H, W)
        # ViT expects 224x224 — resize on the fly
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        feat = self.backbone(x)
        feat = feat.view(B, T, -1).mean(dim=1)
        return self.classifier(feat)

model_vit = ViTStream().to(device)
total_params = sum(p.numel() for p in model_vit.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

Trainable parameters: 5,574,338


In [15]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model_vit.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

EPOCHS = 10
history_vit = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_auc': []}
best_auc = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model_vit, train_loader, optimizer, criterion)
    vl_loss, vl_acc, vl_auc = eval_epoch(model_vit, val_loader, criterion)
    scheduler.step(vl_loss)

    history_vit['train_loss'].append(tr_loss)
    history_vit['val_loss'].append(vl_loss)
    history_vit['train_acc'].append(tr_acc)
    history_vit['val_acc'].append(vl_acc)
    history_vit['val_auc'].append(vl_auc)

    print(f"Epoch {epoch:02d} | Train Loss: {tr_loss:.4f}, Acc: {tr_acc:.4f} | "
          f"Val Loss: {vl_loss:.4f}, Acc: {vl_acc:.4f}, AUC: {vl_auc:.4f}")

    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model_vit.state_dict(), '/kaggle/working/vit_best.pth')
        print(f"  ✓ Best model saved (AUC: {best_auc:.4f})")

Epoch 01 | Train Loss: 0.7570, Acc: 0.5397 | Val Loss: 0.6154, Acc: 0.6667, AUC: 0.6365
  ✓ Best model saved (AUC: 0.6365)
Epoch 02 | Train Loss: 0.7186, Acc: 0.5540 | Val Loss: 0.6512, Acc: 0.6519, AUC: 0.5980
Epoch 03 | Train Loss: 0.5752, Acc: 0.7063 | Val Loss: 0.7200, Acc: 0.7407, AUC: 0.7235
  ✓ Best model saved (AUC: 0.7235)
Epoch 04 | Train Loss: 0.5435, Acc: 0.7857 | Val Loss: 0.5487, Acc: 0.7926, AUC: 0.8348
  ✓ Best model saved (AUC: 0.8348)
Epoch 05 | Train Loss: 0.4207, Acc: 0.8683 | Val Loss: 0.8177, Acc: 0.7185, AUC: 0.9025
  ✓ Best model saved (AUC: 0.9025)
Epoch 06 | Train Loss: 0.5362, Acc: 0.8857 | Val Loss: 1.0197, Acc: 0.7556, AUC: 0.6783
Epoch 07 | Train Loss: 0.3347, Acc: 0.9190 | Val Loss: 1.0636, Acc: 0.7778, AUC: 0.7795
Epoch 08 | Train Loss: 0.1368, Acc: 0.9762 | Val Loss: 0.8312, Acc: 0.8296, AUC: 0.8546
Epoch 09 | Train Loss: 0.0819, Acc: 0.9825 | Val Loss: 1.5737, Acc: 0.8000, AUC: 0.7748
Epoch 10 | Train Loss: 0.1323, Acc: 0.9794 | Val Loss: 0.8058, Acc: 